In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Note on File, Path, Use. Refer to the archived source for the detailed original comment.
import sys
sys.path.append('/kaggle/input/notebooks/fdiiiiii/mri-skimage') 

### Experiment / Training / Validation

This cell documents the brain mri image segmentation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [2]:
%%writefile loss_exp.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceLoss(nn.Module):
    """Dice Loss (原始版本)"""
    def __init__(self, smooth=1e-7):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, y_pred, y_true):
        y_pred = y_pred[:, 0].contiguous().view(-1)
        y_true = y_true[:, 0].contiguous().view(-1)
        intersection = (y_pred * y_true).sum()
        dsc = (2. * intersection + self.smooth) / (y_pred.sum() + y_true.sum() + self.smooth)
        return 1. - dsc


class FocalLoss(nn.Module):
    """Focal Loss (用于处理类别不平衡)"""
    def __init__(self, gamma=2.0, alpha=0.25):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, y_pred, y_true):
        # y_pred: [B, 1, H, W], y_true: [B, 1, H, W]
        y_pred = y_pred[:, 0].contiguous()
        y_true = y_true[:, 0].contiguous()
        
        # Note on Classification. Refer to the archived source for the detailed original comment.
        bce_loss = F.binary_cross_entropy(y_pred, y_true, reduction='none')
        
        # Focal loss
        p_t = y_pred * y_true + (1 - y_pred) * (1 - y_true)
        focal_weight = (1 - p_t) ** self.gamma
        focal_loss = focal_weight * bce_loss
        
        # Alpha weighting
        if self.alpha is not None:
            alpha_t = self.alpha * y_true + (1 - self.alpha) * (1 - y_true)
            focal_loss = alpha_t * focal_loss
        
        return focal_loss.mean()


class DiceBCELoss(nn.Module):
    """Dice Loss + BCE Loss (组合损失)"""
    def __init__(self, dice_weight=1.0, bce_weight=0.5):
        super(DiceBCELoss, self).__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCELoss()

    def forward(self, y_pred, y_true):
        dice = self.dice_loss(y_pred, y_true)
        bce = self.bce_loss(y_pred, y_true)
        return self.dice_weight * dice + self.bce_weight * bce

Writing loss_exp.py


In [3]:
%%writefile train_exp.py

'''
参数化训练脚本 - 支持多种配置对比实验
'''
import os
import pathlib
import time
import json
import tqdm
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

import sys
sys.path.append('/kaggle/input/notebooks/fdiiiiii/mri-skimage') 

from dataset_exp import BrainSegmentationDataset
from transform import apply_affine_transform, get_random_affine_matrix_inv
from utils import log_images, dsc, dataloader_worker_init
from unet_exp import UNet, UNetWithAttention, ResUNet
from loss_exp import DiceLoss, FocalLoss, DiceBCELoss
from transform_exp import ElasticTransform
from transform import apply_affine_transform, get_random_affine_matrix_inv

# Note on Question. Refer to the archived source for the detailed original comment.
def safe_largest_connected_component(img):
    from scipy.ndimage import label
    labeled_array, num_features = label(img, None)
    if num_features == 0:
        return np.zeros_like(img, dtype=bool)
    component_sizes = np.bincount(labeled_array.ravel())[1:]
    return labeled_array == (np.argmax(component_sizes) + 1)

# 覆盖 utils 中的函数
import utils
utils.largest_connected_component = safe_largest_connected_component

def parse_args():
    parser = argparse.ArgumentParser(description='Brain MRI Segmentation Training')
    
    # Note on Data. Refer to the archived source for the detailed original comment.
    parser.add_argument('--data_root', type=str, default=None,
                        help='Data root directory (auto-detect if None)')
    parser.add_argument('--elastic', action='store_true',
                    help='Enable elastic deformation augmentation')
    
    # Note on Network Architecture. Refer to the archived source for the detailed original comment.
    parser.add_argument('--network', type=str, default='unet',
                        choices=['unet', 'attention_unet', 'resunet'],
                        help='Network architecture')
    parser.add_argument('--downsample', type=str, default='maxpool',
                        choices=['maxpool', 'stride_conv'],
                        help='Downsampling mode')
    parser.add_argument('--upsample', type=str, default='convtranspose',
                        choices=['convtranspose', 'bilinear', 'pixelshuffle'],
                        help='Upsampling mode')
    
    # Note on Loss Function. Refer to the archived source for the detailed original comment.
    parser.add_argument('--loss', type=str, default='dice',
                        choices=['dice', 'focal', 'dice_bce'],
                        help='Loss function')
    parser.add_argument('--focal_gamma', type=float, default=2.0,
                        help='Focal loss gamma parameter')
    
    # Note on Optimizer. Refer to the archived source for the detailed original comment.
    parser.add_argument('--optimizer', type=str, default='adam',
                        choices=['adam', 'adamw', 'sgd'],
                        help='Optimizer')
    parser.add_argument('--lr', type=float, default=1e-4,
                        help='Learning rate')
    parser.add_argument('--weight_decay', type=float, default=0,
                        help='Weight decay')
    parser.add_argument('--momentum', type=float, default=0.9,
                        help='SGD momentum')
    
    # Note on Learning Rate. Refer to the archived source for the detailed original comment.
    parser.add_argument('--scheduler', type=str, default='none',
                        choices=['none', 'step', 'plateau', 'cosine'],
                        help='Learning rate scheduler')
    parser.add_argument('--step_size', type=int, default=30,
                        help='StepLR step size')
    parser.add_argument('--plateau_patience', type=int, default=10,
                        help='ReduceLROnPlateau patience')
    
    # Note on Training, Parameters. Refer to the archived source for the detailed original comment.
    parser.add_argument('--epochs', type=int, default=100,
                        help='Number of epochs')
    parser.add_argument('--batch_size', type=int, default=16,
                        help='Batch size')
    parser.add_argument('--vis_freq', type=int, default=10,
                        help='Visualization frequency')
    parser.add_argument('--workers', type=int, default=2,
                        help='DataLoader workers')
    
    # Note on Accuracy. Refer to the archived source for the detailed original comment.
    parser.add_argument('--amp', action='store_true',
                        help='Enable mixed precision training')
    
    # Note on Experiment. Refer to the archived source for the detailed original comment.
    parser.add_argument('--ckpt_name', type=str, default=None,
                        help='Custom checkpoint name')
    parser.add_argument('--comment', type=str, default='',
                        help='Additional comment for experiment name')
    
    return parser.parse_args()


def get_loss_function(args):
    """根据参数返回损失函数"""
    if args.loss == 'dice':
        return DiceLoss()
    elif args.loss == 'focal':
        return FocalLoss(gamma=args.focal_gamma)
    elif args.loss == 'dice_bce':
        return DiceBCELoss()
    else:
        raise ValueError(f"Unknown loss: {args.loss}")


def get_optimizer(model, args):
    """根据参数返回优化器"""
    if args.optimizer == 'adam':
        return optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    elif args.optimizer == 'adamw':
        return optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    elif args.optimizer == 'sgd':
        return optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum,
                        weight_decay=args.weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {args.optimizer}")


def get_scheduler(optimizer, args):
    """根据参数返回学习率调度器"""
    if args.scheduler == 'none':
        return None
    elif args.scheduler == 'step':
        return optim.lr_scheduler.StepLR(optimizer, step_size=args.step_size, gamma=0.1)
    elif args.scheduler == 'plateau':
        return optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                     patience=args.plateau_patience, factor=0.5)
    elif args.scheduler == 'cosine':
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    else:
        raise ValueError(f"Unknown scheduler: {args.scheduler}")


def get_network(args):
    """根据参数返回网络模型"""
    in_channels = BrainSegmentationDataset.in_channels
    out_channels = BrainSegmentationDataset.out_channels
    
    if args.network == 'unet':
        return UNet(in_channels=in_channels, out_channels=out_channels,
                   downsample_mode=args.downsample, upsample_mode=args.upsample)
    elif args.network == 'attention_unet':
        return UNetWithAttention(in_channels=in_channels, out_channels=out_channels,
                                downsample_mode=args.downsample, upsample_mode=args.upsample)
    elif args.network == 'resunet':
        return ResUNet(in_channels=in_channels, out_channels=out_channels,
                      downsample_mode=args.downsample, upsample_mode=args.upsample)
    else:
        raise ValueError(f"Unknown network: {args.network}")


def find_data_root():
    """自动检测数据根目录"""
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if any(d.startswith('TCGA') and os.path.isdir(os.path.join(root, d)) for d in dirs):
                return root
            if any(f.endswith('.tif') for f in files):
                return root
        raise RuntimeError("No brain MRI dataset found under /kaggle/input")
    else:
        return './kaggle_3m'


def plot_curves(ckpt_dir, json_path, config_name):
    """绘制训练曲线"""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    train_losses = data.get('train_loss_per_epoch', [])
    loss_per_epoch = data['loss_per_epoch_per_patient']
    dsc_per_epoch = data['dsc_per_epoch_per_patient']

    epochs = len(loss_per_epoch)
    
    mean_val_loss = [np.mean(epoch_losses) for epoch_losses in loss_per_epoch]
    std_val_loss = [np.std(epoch_losses) for epoch_losses in loss_per_epoch]
    mean_dsc = [np.mean(epoch_dscs) for epoch_dscs in dsc_per_epoch]
    std_dsc = [np.std(epoch_dscs) for epoch_dscs in dsc_per_epoch]

    if len(train_losses) != epochs:
        if len(train_losses) > epochs:
            train_losses = train_losses[:epochs]
        else:
            train_losses += [np.nan] * (epochs - len(train_losses))

    fig, ax1 = plt.subplots(figsize=(12, 6))

    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss', color='black')
    ax1.plot(range(epochs), train_losses, color='orange', linestyle='-', marker='.', label='Train Loss')
    ax1.plot(range(epochs), mean_val_loss, color='red', linestyle='-', marker='.', label='Val Loss')
    ax1.fill_between(range(epochs),
                     np.array(mean_val_loss) - np.array(std_val_loss),
                     np.array(mean_val_loss) + np.array(std_val_loss),
                     color='red', alpha=0.2)
    ax1.tick_params(axis='y')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.5)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('DSC', color=color)
    ax2.plot(range(epochs), mean_dsc, color=color, linestyle='-', marker='.', label='Val DSC')
    ax2.fill_between(range(epochs),
                     np.array(mean_dsc) - np.array(std_dsc),
                     np.array(mean_dsc) + np.array(std_dsc),
                     color=color, alpha=0.2)
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.legend(loc='upper right')

    best_dsc = max(mean_dsc)
    best_epoch = np.argmax(mean_dsc)
    ax2.plot(best_epoch, best_dsc, 'g*', markersize=15, label=f'Best DSC={best_dsc:.4f}')
    ax2.legend(loc='upper right')

    plt.title(f'Training Curves - {config_name}')
    fig.tight_layout()

    plot_path = ckpt_dir / 'loss_dsc_curves.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved curves to {plot_path}')


if __name__ == '__main__':
    args = parse_args()
    
    # Note on Generate. Refer to the archived source for the detailed original comment.
    config_parts = [args.network, args.downsample, args.upsample, args.loss, args.optimizer]
    if args.scheduler != 'none':
        config_parts.append(args.scheduler)
    if args.amp:
        config_parts.append('amp')
    config_name = '_'.join(config_parts)
    
    if args.ckpt_name is None:
        ckpt_name = config_name + args.comment
    else:
        ckpt_name = args.ckpt_name

    # Note on Settings. Refer to the archived source for the detailed original comment.
    if torch.cuda.is_available():
        device = torch.device('cuda:0')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")
    print(f"Configuration: {config_name}")

    # Note on Path, Data. Refer to the archived source for the detailed original comment.
    if args.data_root is None:
        data_root_dir = find_data_root()
    else:
        data_root_dir = args.data_root
    print(f"Data root: {data_root_dir}")

    # Note on Create, Check. Refer to the archived source for the detailed original comment.
    timestamp_str = time.strftime(r'%y%m%d-%H%M%S', time.localtime())
    ckpt_dir = pathlib.Path('./ckpt', f'{timestamp_str} {ckpt_name}')
    print(f'Save directory: {ckpt_dir}')
    os.makedirs(ckpt_dir, exist_ok=True)

    # Note on Save. Refer to the archived source for the detailed original comment.
    with open(ckpt_dir / 'config.json', 'w') as f:
        json.dump(vars(args), f, indent=2)

    # Note on Load, Data. Refer to the archived source for the detailed original comment.
    dataset_train = BrainSegmentationDataset(
        data_root_dir, is_train=True,
        use_elastic=args.elastic   # 根据参数启用
    )
    dataset_valid = BrainSegmentationDataset(data_root_dir, is_train=False)
    loader_train = DataLoader(dataset_train, batch_size=args.batch_size, shuffle=True, 
                              drop_last=True, persistent_workers=True, 
                              pin_memory=True, num_workers=args.workers,
                              worker_init_fn=dataloader_worker_init)
    loader_valid = DataLoader(dataset_valid, batch_size=args.batch_size,
                              drop_last=False, persistent_workers=True,
                              pin_memory=True, num_workers=args.workers,
                              worker_init_fn=dataloader_worker_init)

    # Note on Model, Create. Refer to the archived source for the detailed original comment.
    unet = get_network(args).to(device)
    print(f"Model parameters: {sum(p.numel() for p in unet.parameters()):,}")

    # Note on Loss Function. Refer to the archived source for the detailed original comment.
    criterion = get_loss_function(args)
    
    # Note on Optimizer. Refer to the archived source for the detailed original comment.
    optimizer = get_optimizer(unet, args)
    
    # Note on Learning Rate. Refer to the archived source for the detailed original comment.
    scheduler = get_scheduler(optimizer, args)
    
    # Note on Training, Accuracy. Refer to the archived source for the detailed original comment.
    if args.loss == 'focal' and args.amp:
        print("Warning: Focal Loss is not compatible with AMP. Disabling AMP for this run.")
        args.amp = False
        scaler = None
    else:
        scaler = torch.cuda.amp.GradScaler() if args.amp and torch.cuda.is_available() else None
    if args.amp and not torch.cuda.is_available():
        print("Warning: AMP requires CUDA, disabling.")

    summarywriter = SummaryWriter(pathlib.Path(ckpt_dir, 'logs'))

    # Note on Training, Loop. Refer to the archived source for the detailed original comment.
    loss_per_epoch_per_patient = []
    dsc_per_epoch_per_patient = []
    train_losses_per_epoch = []

    best_validation_dsc = 0.0

    for epoch in range(args.epochs):
        # Note on Training. Refer to the archived source for the detailed original comment.
        unet.train()
        losses = []
        epoch_loader_train = tqdm.tqdm(loader_train, desc=f'E#{epoch}', postfix='init')
        
        for i, (xs, ys_true, gs_inv) in enumerate(epoch_loader_train):
            xs = xs.to(device)
            ys_true = ys_true.to(device)
            gs_inv = gs_inv.to(device)

            with torch.no_grad():
                xs = apply_affine_transform(xs, gs_inv, interpolation='bilinear', edge_fill_value=0.)
                ys_true = apply_affine_transform(ys_true, gs_inv, interpolation='nearest', edge_fill_value=0.)

            optimizer.zero_grad()
            
            # Note on Accuracy. Refer to the archived source for the detailed original comment.
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    ys_pred = unet(xs)
                    loss = criterion(ys_pred, ys_true)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                ys_pred = unet(xs)
                loss = criterion(ys_pred, ys_true)
                loss.backward()
                optimizer.step()

            losses.append(loss.item())
            epoch_loader_train.set_postfix_str(f'L={np.mean(losses):.4f}')

        mean_train_loss = np.mean(losses)
        train_losses_per_epoch.append(mean_train_loss)
        summarywriter.add_scalar('train/loss', mean_train_loss, epoch)

        # Note on Validation. Refer to the archived source for the detailed original comment.
        unet.eval()
        
        n_slices_per_patient = np.bincount(dataset_valid.patient_slice_index[..., 0])
        i0_slice_per_patient = np.cumsum(n_slices_per_patient) - n_slices_per_patient
        
        loss_per_patient = []
        dsc_per_patient = []
        
        for i_patient, (i0_slice, n_slices) in enumerate(zip(i0_slice_per_patient, n_slices_per_patient)):
            patient_name = dataset_valid.patients[i_patient]
            
            patient_losses = []
            patient_ys_pred = []
            patient_ys_true = []
            
            for i_slice in range(i0_slice, i0_slice + n_slices):
                x, y_true, g_inv = dataset_valid[i_slice]
                xs = x[None].to(device)
                ys_true = y_true[None].to(device)

                with torch.no_grad():
                    if scaler is not None:
                        with torch.cuda.amp.autocast():
                            ys_pred = unet(xs)
                            loss = criterion(ys_pred, ys_true)
                    else:
                        ys_pred = unet(xs)
                        loss = criterion(ys_pred, ys_true)

                    patient_losses.append(loss.item())
                    patient_ys_pred.append(ys_pred.squeeze(0).detach().cpu().numpy())
                    patient_ys_true.append(ys_true.squeeze(0).detach().cpu().numpy())

                    if (epoch % args.vis_freq == 0) or (epoch == args.epochs - 1):
                        img, = log_images(xs, ys_true, ys_pred)
                        summarywriter.add_image(f'val/images/{i_patient}_{patient_name}/{i_slice}', 
                                               img, epoch, dataformats='HWC')

            loss_per_patient.append(float(np.mean(patient_losses)))
            dsc_per_patient.append(float(dsc(np.stack(patient_ys_pred), np.stack(patient_ys_true))))

        loss_per_epoch_per_patient.append(loss_per_patient)
        dsc_per_epoch_per_patient.append(dsc_per_patient)

        mean_val_loss = np.mean(loss_per_patient)
        mean_dsc = np.mean(dsc_per_patient)
        
        summarywriter.add_scalar('val/loss', mean_val_loss, epoch)
        summarywriter.add_scalar('val/dsc', mean_dsc, epoch)
        summarywriter.add_scalar('train/learning_rate', optimizer.param_groups[0]['lr'], epoch)
        
        print(f'  val loss: {mean_val_loss:.4f}, val dsc: {mean_dsc:.4f}')

        # Note on Learning Rate. Refer to the archived source for the detailed original comment.
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(mean_val_loss)
            else:
                scheduler.step()

        # Note on Model, Save. Refer to the archived source for the detailed original comment.
        if mean_dsc > best_validation_dsc:
            best_validation_dsc = mean_dsc
            torch.save(unet.state_dict(), ckpt_dir / 'unet.pt')
            print(f'  New best model saved (DSC: {best_validation_dsc:.4f})')

    print(f'Best validation mean DSC: {best_validation_dsc:.6f}')

    # Note on Training, Save. Refer to the archived source for the detailed original comment.
    with open(ckpt_dir / 'losses_dsc_values.json', 'wt', encoding='utf-8') as f:
        json.dump({
            'best_dsc': best_validation_dsc,
            'patient_names': dataset_valid.patients,
            'train_loss_per_epoch': train_losses_per_epoch,
            'loss_per_epoch_per_patient': loss_per_epoch_per_patient,
            'dsc_per_epoch_per_patient': dsc_per_epoch_per_patient,
        }, f, indent=2, ensure_ascii=False)

    # Note on Plot. Refer to the archived source for the detailed original comment.
    plot_curves(ckpt_dir, ckpt_dir / 'losses_dsc_values.json', config_name)
    summarywriter.close()

Writing train_exp.py


In [4]:
%%writefile unet_exp.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class UNet(nn.Module):
    """标准 UNet，支持参数化下采样/上采样"""
    def __init__(self, in_channels=3, out_channels=1, init_features=32,
                 downsample_mode='maxpool', upsample_mode='convtranspose'):
        super(UNet, self).__init__()

        def _block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(),
            )

        # 编码器
        enc_channels = [(in_channels, init_features),
                        (init_features, init_features*2),
                        (init_features*2, init_features*4),
                        (init_features*4, init_features*8)]

        self.enc_blocks = nn.ModuleList()
        self.down_ops = nn.ModuleList()

        for i, (in_c, out_c) in enumerate(enc_channels):
            self.enc_blocks.append(_block(in_c, out_c))
            if i < len(enc_channels) - 1:
                if downsample_mode == 'maxpool':
                    down = nn.MaxPool2d(2, 2)
                elif downsample_mode == 'stride_conv':
                    down = nn.Conv2d(out_c, out_c, kernel_size=3, stride=2, padding=1)
                else:
                    raise ValueError(f"Unsupported downsample_mode: {downsample_mode}")
                self.down_ops.append(down)
            else:
                self.down_ops.append(None)

        self.bottleneck = _block(init_features*8, init_features*16)

        # 解码器
        dec_channels = [(init_features*16, init_features*8),
                        (init_features*8, init_features*4),
                        (init_features*4, init_features*2),
                        (init_features*2, init_features)]

        self.up_ops = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()

        for i, (in_c, out_c) in enumerate(dec_channels):
            if upsample_mode == 'convtranspose':
                up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)
            elif upsample_mode == 'bilinear':
                # 修复：先卷积降通道，再上采样
                up = nn.Sequential(
                    nn.Conv2d(in_c, out_c, kernel_size=1),
                    nn.UpsamplingBilinear2d(scale_factor=2)
                )
            elif upsample_mode == 'pixelshuffle':
                up = nn.Sequential(
                    nn.Conv2d(in_c, out_c * 4, kernel_size=1),
                    nn.PixelShuffle(upscale_factor=2)
                )
            else:
                raise ValueError(f"Unsupported upsample_mode: {upsample_mode}")
            self.up_ops.append(up)

            skip_in = enc_channels[-1-i][1]
            self.dec_blocks.append(_block(out_c + skip_in, out_c))

        self.conv = nn.Conv2d(init_features, out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc_blocks, self.down_ops):
            x = enc(x)
            skips.append(x)
            if down is not None:
                x = down(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.up_ops, self.dec_blocks, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        return torch.sigmoid(self.conv(x))

Writing unet_exp.py


In [5]:
%%writefile dataset_exp.py

import os
import pathlib
import itertools
import random
import numpy as np
import torch
import skimage.io
import torch.utils.data
from transform import apply_affine_transform, get_random_affine_matrix_inv
from transform_exp import ElasticTransform   # 导入弹性形变


class BrainSegmentationDataset(torch.utils.data.Dataset):
    in_channels = 3
    out_channels = 1

    def __init__(
        self,
        images_dir: str | pathlib.Path,
        is_train: bool = True,
        validation_cases: int = 10,
        use_elastic: bool = False,          # 是否启用弹性形变
        elastic_alpha: float = 34,
        elastic_sigma: float = 4,
        elastic_p: float = 0.5,
    ):
        self.is_train = is_train
        self.use_elastic = use_elastic and is_train   # 仅在训练时使用

        if self.use_elastic:
            self.elastic = ElasticTransform(alpha=elastic_alpha, sigma=elastic_sigma, p=elastic_p)

        images_dir = pathlib.Path(images_dir)

        all_scan_img_paths = [scan_img_path for scan_img_path in images_dir.rglob('*.tif')
            if not scan_img_path.stem.endswith('_mask')]
        all_scan_img_paths = sorted(all_scan_img_paths, key=
            lambda scan_img_path: (scan_img_path.parent.name, int(scan_img_path.stem.rsplit('_', maxsplit=1)[1])))
        assert len(all_scan_img_paths) > 0, f'No images found in {images_dir}'

        all_patient_to_scan_img_paths = {patient: list(scan_img_paths)
            for patient, scan_img_paths in
                itertools.groupby(all_scan_img_paths, key=lambda scan_img_path: scan_img_path.parent.name)}

        self.patients = list(all_patient_to_scan_img_paths.keys())

        # Note on Training, Validation. Refer to the archived source for the detailed original comment.
        random.seed(42)
        validation_patients = sorted(random.sample(self.patients, k=validation_cases))
        if is_train:
            self.patients = sorted(set(self.patients).difference(validation_patients))
        else:
            self.patients = validation_patients

        # Note on Image, Load. Refer to the archived source for the detailed original comment.
        subset_name = 'train' if is_train else 'validation'
        print(f'Reading {sum(len(all_patient_to_scan_img_paths[patient]) for patient in self.patients)} images from {subset_name} set ...')
        self.patient_scans_float32 = []
        self.patient_masks_float32 = []
        for patient in self.patients:
            scans_uint8 = []
            masks_uint8 = []
            for scan_img_path in all_patient_to_scan_img_paths[patient]:
                scans_uint8.append(skimage.io.imread(scan_img_path))
                masks_uint8.append(skimage.io.imread(scan_img_path.with_stem(scan_img_path.stem + '_mask'), as_gray=True)[..., np.newaxis])

            self.patient_scans_float32.append((np.stack(scans_uint8) / 255).astype(np.float32))   # (n_img, 256, 256, 3)
            self.patient_masks_float32.append((np.stack(masks_uint8) / 255).astype(np.float32))   # (n_img, 256, 256, 1)

        # Note on Weights. Refer to the archived source for the detailed original comment.
        self.slice_weights = [np.sum(masks_float32, axis=(-1, -2, -3)) for masks_float32 in self.patient_masks_float32]
        self.slice_weights = [(s + (s.sum() * 0.1 / len(s))) / (s.sum() * 1.1) for s in self.slice_weights]

        # 全局索引
        self.patient_n_imgs = [scans_float32.shape[0] for scans_float32 in self.patient_scans_float32]
        self.patient_slice_index = np.stack([
            np.repeat(np.arange(len(self.patient_n_imgs)), self.patient_n_imgs),
            np.concatenate([np.arange(n_imgs) for n_imgs in self.patient_n_imgs]),
        ], axis=-1)

    def __len__(self):
        return len(self.patient_slice_index)

    def __getitem__(self, idx: int):
        if self.is_train:
            i_patient = np.random.randint(len(self.patients))
            slice_n = np.random.choice(
                range(self.patient_n_imgs[i_patient]),
                p=self.slice_weights[i_patient])
        else:
            i_patient, slice_n = self.patient_slice_index[idx]

        scan_float32 = self.patient_scans_float32[i_patient][slice_n]   # (H,W,C)
        mask_float32 = self.patient_masks_float32[i_patient][slice_n]   # (H,W,1)

        # Note on Training. Refer to the archived source for the detailed original comment.
        if self.use_elastic:
            # 将 mask 从 (H,W,1) 转为 (H,W) 便于处理
            mask_2d = mask_float32.squeeze(-1)
            scan_float32, mask_2d = self.elastic(scan_float32, mask_2d)
            mask_float32 = mask_2d[..., np.newaxis]   # 恢复 (H,W,1)

        # 转置为 (C, H, W)
        scan_float32 = scan_float32.transpose(2, 0, 1)
        mask_float32 = mask_float32.transpose(2, 0, 1)

        scan = torch.from_numpy(scan_float32.astype(np.float32))
        mask = torch.from_numpy(mask_float32.astype(np.float32))

        if self.is_train:
            g_inv = get_random_affine_matrix_inv(
                reflection=True,
                angle_range_deg=(-45., 45.),
                scale_range=(0.75, 1.33),
                translate_range=(-0.2, 0.2))
        else:
            g_inv = torch.eye(3, dtype=torch.float32)

        return scan, mask, g_inv

Writing dataset_exp.py


In [6]:
%%writefile transform_exp.py

import numpy as np
from scipy.ndimage import map_coordinates, gaussian_filter


def elastic_transform(image, mask, alpha, sigma, random_state=None):
    """
    对图像和掩码同时应用弹性形变，保证一致性
    image: (H, W, C) 或 (H, W)
    mask:  (H, W) 或 (H, W, 1)
    """
    if random_state is None:
        random_state = np.random.RandomState(None)
    shape = image.shape[:2]  # (H, W)
    dx = gaussian_filter(random_state.randn(*shape), sigma, mode='constant', cval=0) * alpha
    dy = gaussian_filter(random_state.randn(*shape), sigma, mode='constant', cval=0) * alpha
    x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]), indexing='xy')
    indices = (y + dy, x + dx)

    # Note on Image. Refer to the archived source for the detailed original comment.
    if image.ndim == 3:
        warped_image = np.zeros_like(image)
        for c in range(image.shape[2]):
            warped_image[..., c] = map_coordinates(image[..., c], indices, order=1, mode='reflect')
    else:
        warped_image = map_coordinates(image, indices, order=1, mode='reflect')
    
    # 处理掩码 (最近邻插值保持二值性)
    if mask.ndim == 3:
        warped_mask = np.zeros_like(mask)
        for c in range(mask.shape[2]):
            warped_mask[..., c] = map_coordinates(mask[..., c], indices, order=0, mode='reflect')
    else:
        warped_mask = map_coordinates(mask, indices, order=0, mode='reflect')
    
    return warped_image, warped_mask


class ElasticTransform:
    def __init__(self, alpha=34, sigma=4, p=0.5):
        self.alpha = alpha
        self.sigma = sigma
        self.p = p

    def __call__(self, image, mask):
        if np.random.random() < self.p:
            return elastic_transform(image, mask, self.alpha, self.sigma)
        return image, mask

Writing transform_exp.py


In [7]:
# Note on Loss Function. Refer to the archived source for the detailed original comment.
!python train_exp.py --loss focal --ckpt_name focal

Using device: cuda:0
Configuration: unet_maxpool_convtranspose_focal_adam
Data root: /kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m
Save directory: ckpt/260705-153859 focal
Reading 3574 images from train set ...
Reading 355 images from validation set ...
Model parameters: 7,763,041
E#0: 100%|██████████████████████████| 223/223 [01:17<00:00,  2.87it/s, L=0.0776]
  val loss: 0.0509, val dsc: 0.5565
  New best model saved (DSC: 0.5565)
E#1: 100%|██████████████████████████| 223/223 [01:22<00:00,  2.71it/s, L=0.0412]
  val loss: 0.0343, val dsc: 0.5603
  New best model saved (DSC: 0.5603)
E#2: 100%|██████████████████████████| 223/223 [01:31<00:00,  2.44it/s, L=0.0275]
  val loss: 0.0213, val dsc: 0.6078
  New best model saved (DSC: 0.6078)
E#3: 100%|██████████████████████████| 223/223 [01:33<00:00,  2.38it/s, L=0.0195]
  val loss: 0.0151, val dsc: 0.5872
E#4: 100%|██████████████████████████| 223/223 [01:34<00:00,  2.36it/s, L=0.0145]
  val loss: 0.0114, val dsc: 0.6238
  

In [8]:
# 上采样对比：Bilinear
!python train_exp.py --upsample bilinear --ckpt_name up_bilinear

Using device: cuda:0
Configuration: unet_maxpool_bilinear_dice_adam
Data root: /kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m
Save directory: ckpt/260705-182618 up_bilinear
Reading 3574 images from train set ...
Reading 355 images from validation set ...
Model parameters: 7,240,801
E#0: 100%|██████████████████████████| 223/223 [01:28<00:00,  2.53it/s, L=0.8449]
  val loss: 0.9446, val dsc: 0.3256
  New best model saved (DSC: 0.3256)
E#1: 100%|██████████████████████████| 223/223 [01:27<00:00,  2.56it/s, L=0.8017]
  val loss: 0.9336, val dsc: 0.6959
  New best model saved (DSC: 0.6959)
E#2: 100%|██████████████████████████| 223/223 [01:26<00:00,  2.59it/s, L=0.7469]
  val loss: 0.9164, val dsc: 0.7108
  New best model saved (DSC: 0.7108)
E#3: 100%|██████████████████████████| 223/223 [01:26<00:00,  2.59it/s, L=0.6862]
  val loss: 0.9000, val dsc: 0.6178
E#4: 100%|██████████████████████████| 223/223 [01:25<00:00,  2.60it/s, L=0.6169]
  val loss: 0.8730, val dsc: 0.7557
  

In [9]:
# 全部对比
!python train_exp.py --elastic --loss focal --optimizer adamw --amp --ckpt_name ultimate

Using device: cuda:0
Configuration: unet_maxpool_convtranspose_focal_adamw_amp
Data root: /kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m
Save directory: ckpt/260705-210049 ultimate
Reading 3574 images from train set ...
Reading 355 images from validation set ...
Model parameters: 7,763,041
E#0: 100%|██████████████████████████| 223/223 [01:34<00:00,  2.37it/s, L=0.0539]
  val loss: 0.0499, val dsc: 0.4987
  New best model saved (DSC: 0.4987)
E#1: 100%|██████████████████████████| 223/223 [01:34<00:00,  2.36it/s, L=0.0268]
  val loss: 0.2223, val dsc: 0.5102
  New best model saved (DSC: 0.5102)
E#2: 100%|██████████████████████████| 223/223 [01:33<00:00,  2.38it/s, L=0.0195]
  val loss: 0.0215, val dsc: 0.4911
E#3: 100%|██████████████████████████| 223/223 [01:33<00:00,  2.38it/s, L=0.0151]
  val loss: 0.0108, val dsc: 0.6257
  New best model saved (DSC: 0.6257)
E#4: 100%|██████████████████████████| 223/223 [01:33<00:00,  2.38it/s, L=0.0118]
  val loss: 0.0074, val dsc: 0